#### 第五章 误差反向传播法
数值微分简单容易实现但是计算上比较费时间 我们来学习一个能够高效计算权重梯度的方法 误差反向传播法
5.1 计算图

5.4 简单层的实现

In [7]:
import numpy as np


#5.4.1乘法层的实现
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None
    def forward(self, x, y):
        # 前向传播的时候会保存值 然后反向传播的时候用
        self.x = x
        self.y = y
        out = x * y
        return out
    #用前面传来的导数经过这个乘法节点 反向传播
    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x#翻转x和y
        return dx, dy


In [8]:
#使用这个代码完成正向传播
apple = 100
apple_num = 2
tax = 1.1
#layer
mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()
#forward
"""我的理解是每次经过正向传播对于一个特定的输入 ，那么反向传播求导数的值也会是固定的"""
apple_price = mul_apple_layer.forward(apple, apple_num)
price = mul_tax_layer.forward(apple_price, tax)
print("price:", price)

price: 220.00000000000003


In [9]:
#backward
#这里要注意 backward的参数中需要输入关于正向传播时的输出变量的导数， 那不就是1吗 自己关于自己的导数（最末尾的那一份）
#
dprice = 1
#解释一下各个变量：
#dapple_price:最终的价格关于苹果的售价的导数
#dtax：最终的价格关于税率的导数
#dapple：最终的价格关于苹果的数量的导数
#dtax：最终的价格关于税率的导数
dapple_price, dtax = mul_tax_layer.backward(dprice)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)
print("dprice:", dprice)
print("dtax:", dtax)
print("dapple:", dapple)
print("dapple_price:", dapple_price)

dprice: 1
dtax: 200
dapple: 2.2
dapple_price: 1.1


In [10]:
#5.4.2 加法层的实现
class AddLayer:
    def __init__(self):
        pass
    def forward(self, x, y):
        return x + y
    """加法层求导，实际上没变啊 但是这里还是变成了一个浮点数"""
    def backward(self, dout):
        dx = dout * 1.0
        dy = dout * 1.0
        return dx, dy


In [11]:
#5.4.1乘法层的实现
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None
    def forward(self, x, y):
        # 前向传播的时候会保存值 然后反向传播的时候用
        self.x = x
        self.y = y
        out = x * y
        return out
    def backward(self, dout):
        dx = dout * self.y
        dy = dout * self.x#翻转x和y
        return dx, dy
#5.4.2 加法层的实现
class AddLayer:
    def __init__(self):
        pass
    def forward(self, x, y):
        return x + y
    def backward(self, dout):
        dx = dout * 1.0
        dy = dout * 1.0
        return dx, dy

#实现买苹果和橘子
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()
#forward
apple_price = mul_apple_layer.forward(apple, apple_num)
orange_price = mul_orange_layer.forward(orange, orange_num)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)
price = mul_tax_layer.forward(all_price, tax)
#backward
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)
print("price:", price)
print(dapple_num, dapple, dorange, dorange_num, dtax)

price: 715.0000000000001
110.00000000000001 2.2 3.3000000000000003 165.0 650


In [12]:
#用Python实现图5-17的计算图的过程如下所示
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1
#layer
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()
#forward
apple_price = mul_apple_layer.forward(apple, apple_num)
orange_price = mul_orange_layer.forward(orange, orange_num)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)
price = mul_tax_layer.forward(all_price, tax)
#backward
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)
print("price:", price)
print(dapple_num, dapple, dorange, dorange_num, dtax)

In [18]:
class Relu:
    def __init__(self):
        self.mask = None
    def forward(self, x):
        #先找出x<=0的部分的序号
        self.mask = (x <= 0)
        #然后对于这一部分 变成0 其他部分不变
        out = x.copy
        out[self.mask] = 0
        return out
    def backward(self, dout):
        #<=0的部分保持求导就是0 其他部分因为导数是1 所以相当于还是dout，这个花式索引到底在干什么
        #以为在滑轮滑吗
        #所以这里的self.mask存的是 输入是否小于0
        #求导数就是
        dout[self.mask] = 0
        dx = dout
        return dx

In [14]:
import numpy as np
x = np.array([[1.0, -0.5],[-2.0, 3.0]])
print(x)
mask = (x <= 0)
print(mask)

[[ 1.  -0.5]
 [-2.   3. ]]
[[False  True]
 [ True False]]


In [19]:
#5.2.2 sigmoid层
class Sigmoid:
    def __init__(self):
        self.out = None
    def forward(self, x):
        self.out = x / (1 + np.exp(-x))
        return self.out
    def backward(self,dout):
        # 这个还是比较好求导数的 主要是记一下当前的状态的参数，看一下求导数跟哪些值有关
        dx = dout * (1.0 - self.out) * self.out
        return dx

In [20]:
#5.6.1 Affine 层
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None
    #前向传播层
    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b
        return out
    #这里的导数就是转置，书中没有给出详细的解释
    #求一个就是另外一个的转置
    #我们要做的就是更新参数
    #db这里是因为 前面做运算的时候把每一个参数都加了一个b，所以求导的时候 前面的都是b的系数 所以要把他们按照行加起来
    def backward(self, dout):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx

#### 5.6.3交叉熵损失函数  总之就是很复杂的计算了一堆我去 但是结果很漂亮 正是因为这个很漂亮的结果才设计了这个交叉熵损失函数的
之前还在考虑为什么要设计这个交叉熵损失函数

In [22]:
def softmax(a):
    if a.ndim == 2:  # batch
        c = np.max(a, axis=1, keepdims=True)
        exp_a = np.exp(a - c)
        sum_exp_a = np.sum(exp_a, axis=1, keepdims=True)
        y = exp_a / sum_exp_a
    else:  # 单样本
        c = np.max(a)
        exp_a = np.exp(a - c)
        sum_exp_a = np.sum(exp_a)
        y = exp_a / sum_exp_a
    return y
 #mini batch版交叉熵误差的实现
def cross_entropy_error(y, t):
    delta = 1e-7
    if y.ndim == 1:
        t = t.reshape(1,t.size)
        y = y.reshape(1,y.size)
    batch_size = y.shape[0]
    return -np.sum(np.log(y[np.arange(batch_size),t] + delta))/ batch_size

In [23]:
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None
        self.y = None
        self.t = None
    def forward(self, x, t):
        self.y = softmax(x)
        self.t = t
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss
    def backward(self, dout):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size

        return dx

##### 5.7 误差反向传播法的实现 为了方便调试 同样在test中实现